# Project 3 — Food & Restaurant Services: AI Demand Forecasting
# Executive problem statement and objectives

"""
Build an end-to-end demand forecasting pipeline for restaurant operations using the
Kaggle dataset `gauravsahani/food-demand-prediction-dataset`.
"""

In [ ]:
# ==============================
# 1) Install & Import Dependencies
# ==============================

# Run these in a notebook cell to install required packages if missing.
!pip install -q kagglehub pandas numpy scikit-learn xgboost matplotlib seaborn plotly joblib

# Imports
import os
import warnings
warnings.filterwarnings('ignore')

import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import joblib

print('Imports complete')

In [ ]:
# ==============================
# 2) Download Dataset from KaggleHub
# ==============================

# Download latest version of the dataset
path = kagglehub.dataset_download("gauravsahani/food-demand-prediction-dataset")
print("Path to dataset files:", path)

# List files
for f in os.listdir(path):
    print('-', f)

In [ ]:
# ==============================
# 3) Load and Inspect Raw Data
# ==============================

# Attempt to find a main CSV in the dataset folder. Adjust filename if dataset differs.
csv_files = [f for f in os.listdir(path) if f.lower().endswith('.csv')]
print('CSV files found:', csv_files)

# Load first CSV found
if len(csv_files) == 0:
    raise FileNotFoundError('No CSV file found in dataset path')

raw_df = pd.read_csv(os.path.join(path, csv_files[0]))
print('Raw shape:', raw_df.shape)
raw_df.head()

# Progress update: basic notebook skeleton, installs, dataset download, and raw load cells added.
# Next: add cleaning, EDA, feature engineering, modeling, evaluation, and save/export cells.

In [ ]:
# ==============================
# 4) Clean, Parse, and Sort Time-Series Data
# ==============================

# Identify a datetime column — try common names
possible_date_cols = [c for c in raw_df.columns if 'date' in c.lower() or 'day' in c.lower()]
print('Possible date columns:', possible_date_cols)

# If dataset already has a date column, use it. Otherwise attempt to infer an index.
date_col = possible_date_cols[0] if len(possible_date_cols) > 0 else None

if date_col is None:
    # Try to find a datetime-like first column
    date_col = raw_df.columns[0]

print('Using date column:', date_col)
raw_df[date_col] = pd.to_datetime(raw_df[date_col], errors='coerce')

# Drop rows without a valid date
raw_df = raw_df.dropna(subset=[date_col]).copy()
raw_df = raw_df.sort_values(by=date_col).reset_index(drop=True)

# Infer target column: try 'sales', 'demand', 'quantity', 'orders', 'num_orders'
possible_targets = [c for c in raw_df.columns if any(k in c.lower() for k in ['sales','demand','quantity','orders','num_orders','total'])]
print('Possible targets:', possible_targets)

target_col = possible_targets[0] if len(possible_targets) > 0 else None
if target_col is None:
    raise ValueError('No obvious target column found. Inspect raw_df.columns manually.')

print('Using target column:', target_col)

# Aggregate to daily level if data is transactional
raw_df['date'] = raw_df[date_col].dt.date

daily = raw_df.groupby('date')[target_col].sum().reset_index()

daily['date'] = pd.to_datetime(daily['date'])

daily = daily.set_index('date').asfreq('D')  # ensure continuous daily index (NaNs for missing days)

# Fill missing days with 0 or interpolation — choose forward fill or 0 depending on domain
if daily[target_col].isna().sum() > 0:
    daily[target_col] = daily[target_col].fillna(0)

daily = daily.rename(columns={target_col: 'sales'})
print('Daily shape:', daily.shape)
daily.head()

In [ ]:
# ==============================
# 5) Time-Series Exploratory Data Analysis
# ==============================

# Quick plots
plt.figure(figsize=(12,4))
plt.plot(daily.index, daily['sales'], color='tab:blue')
plt.title('Daily Sales')
plt.ylabel('Sales')
plt.xlabel('Date')
plt.grid(alpha=0.2)
plt.show()

# Weekday pattern
daily['dow'] = daily.index.dayofweek
sns.boxplot(x='dow', y='sales', data=daily.reset_index())
plt.title('Sales by Day of Week (0=Mon)')
plt.show()

# Monthly pattern
daily['month'] = daily.index.month
sns.boxplot(x='month', y='sales', data=daily.reset_index())
plt.title('Sales by Month')
plt.show()

# ACF plot (quick)
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(daily['sales'].fillna(0), lags=30)
plt.show()

In [ ]:
# ==============================
# 6) Create Calendar, Lag, and Rolling Features
# ==============================

def create_features(df, target_col='sales'):
    df = df.copy()
    df['dow'] = df.index.dayofweek
    df['month'] = df.index.month
    df['is_weekend'] = df['dow'].isin([5,6]).astype(int)

    # Lags
    for l in [1,7,14,30]:
        df[f'lag_{l}'] = df[target_col].shift(l)

    # Rolling statistics
    for w in [7,14,30]:
        df[f'roll_mean_{w}'] = df[target_col].shift(1).rolling(w).mean()
        df[f'roll_std_{w}'] = df[target_col].shift(1).rolling(w).std()

    # EMA and momentum
    df['ema_7'] = df[target_col].shift(1).ewm(span=7, adjust=False).mean()
    df['momentum_7'] = df[target_col].shift(1) - df[target_col].shift(8)

    # Drop rows with NaN features (from lags)
    df = df.dropna()
    return df

feature_df = create_features(daily)
print('Feature frame shape:', feature_df.shape)
feature_df.head()

In [ ]:
# ==============================
# 7) Train/Test Split (rolling / time-aware)
# ==============================

# We'll use a simple split: last 14 days as test
test_size_days = 14
train_df = feature_df.iloc[:-test_size_days]
test_df = feature_df.iloc[-test_size_days:]

X_train = train_df.drop(columns=['sales'])
y_train = train_df['sales']
X_test = test_df.drop(columns=['sales'])
y_test = test_df['sales']

print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

In [ ]:
# ==============================
# 8) Baseline Models + XGBoost with TimeSeriesSplit GridSearch
# ==============================

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV

# Define simple baseline
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=200, random_state=42)

# XGBoost param grid
xgb = XGBRegressor(objective='reg:squarederror', random_state=42, n_jobs=4)
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3,6],
    'learning_rate': [0.05, 0.1]
}

tscv = TimeSeriesSplit(n_splits=5)

grid = GridSearchCV(xgb, param_grid, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=1)

# Fit baseline and tuned xgb
print('Fitting LinearRegression')
lr.fit(X_train, y_train)
print('Fitting RandomForest')
rf.fit(X_train, y_train)
print('Fitting XGB GridSearch')
grid.fit(X_train, y_train)

best_xgb = grid.best_estimator_
print('Best XGB params:', grid.best_params_)

# Predictions
pred_lr = lr.predict(X_test)
pred_rf = rf.predict(X_test)
pred_xgb = best_xgb.predict(X_test)

# Evaluation
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

print('LR MAE:', mae(y_test, pred_lr), 'RMSE:', rmse(y_test, pred_lr))
print('RF MAE:', mae(y_test, pred_rf), 'RMSE:', rmse(y_test, pred_rf))
print('XGB MAE:', mae(y_test, pred_xgb), 'RMSE:', rmse(y_test, pred_xgb))

# Choose best
results = {
    'LinearRegression': {'mae': mae(y_test, pred_lr), 'rmse': rmse(y_test, pred_lr)},
    'RandomForest': {'mae': mae(y_test, pred_rf), 'rmse': rmse(y_test, pred_rf)},
    'Tuned XGBoost': {'mae': mae(y_test, pred_xgb), 'rmse': rmse(y_test, pred_xgb)}
}

best_name = min(results.keys(), key=lambda k: results[k]['mae'])
print('Best on MAE:', best_name)

In [ ]:
# ==============================
# 9) Feature Importance and Diagnostic Plots
# ==============================

# Feature importance from best_xgb
fi = pd.DataFrame({'feature': X_train.columns, 'importance': best_xgb.feature_importances_})
fi = fi.sort_values('importance', ascending=False)

plt.figure(figsize=(8,6))
sns.barplot(x='importance', y='feature', data=fi.head(20))
plt.title('XGBoost Feature Importance')
plt.show()

# Actual vs Predicted for XGB
plt.figure(figsize=(12,5))
plt.plot(y_test.index, y_test, label='Actual')
plt.plot(y_test.index, pred_xgb, label='XGB Pred')
plt.legend()
plt.title('Actual vs Predicted (XGB)')
plt.show()

In [ ]:
# ==============================
# 10) Save model and export forecasts
# ==============================

os.makedirs('artifacts', exist_ok=True)
joblib.dump(best_xgb, 'artifacts/best_xgb.joblib')

# Export test predictions
out = pd.DataFrame({
    'date': y_test.index,
    'actual': y_test.values,
    'pred_xgb': pred_xgb
})
out.to_csv('artifacts/xgb_test_forecast.csv', index=False)

print('Saved model and forecasts to artifacts/')